# Day 2 · 이론 ④ — 교차 검증과 하이퍼파라미터 탐색 · 추가 실습

**공식 실습:** `5-2.ipynb`  
**추가 실습:** `5-2_extra_practice.ipynb`

**실습 A(교안 3-5) → B 준비(교안 5-2) → 실습 B(교안 5-5)** 순서로 실행합니다. 설명·핵심 질문·정답은 강의교안을 참고합니다.

## 0. 공통 실습 환경 준비

교안 2-1·2-2와 동일한 와인 데이터 및 `train_test_split(test_size=0.2, random_state=42)`을 사용합니다. CSV 다운로드에는 인터넷 연결이 필요합니다.

In [ ]:
import pandas as pd

wine = pd.read_csv('https://bit.ly/wine_csv_data')

data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

In [ ]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42
)

print(train_input.shape, test_input.shape)

## 실습 A — 교차 검증으로 트리 깊이 직접 선택하기

교안 3-5. 동일한 5-폴드로 `max_depth`별 훈련·검증 정확도와 검증 표준편차를 비교합니다.

### A-1. 깊이별 5-폴드 교차 검증

In [ ]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

depth_candidates = [1, 2, 3, 5, 10, None]  # None은 깊이 제한 없음

# 모든 후보에 같은 5-폴드 분할을 사용
cv5 = StratifiedKFold(n_splits=5)

depth_results = []

for depth in depth_candidates:
    dt_candidate = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    # 폴드별 훈련 점수도 함께 받기 위해 return_train_score=True
    scores_a = cross_validate(
        dt_candidate,
        train_input,
        train_target,
        cv=cv5,
        return_train_score=True
    )

    depth_results.append({
        'max_depth': '제한 없음' if depth is None else str(depth),
        '평균 훈련 정확도': np.mean(scores_a['train_score']),
        '평균 검증 정확도': np.mean(scores_a['test_score']),
        '검증 표준편차': np.std(scores_a['test_score'])
    })

results_a = pd.DataFrame(depth_results)
print(results_a.round(4).to_string(index=False))

### A-2. 평균 검증 정확도 내림차순 정렬

In [ ]:
ranked_a = results_a.sort_values(
    '평균 검증 정확도',
    ascending=False
)

print(ranked_a.round(4).to_string(index=False))

### A-3. 검증 결과로 트리 깊이 선택

In [ ]:
best_row = ranked_a.iloc[0]

print('선택한 깊이:', best_row['max_depth'])
print('평균 검증 정확도:', round(best_row['평균 검증 정확도'], 4))
print('검증 표준편차:', round(best_row['검증 표준편차'], 4))

### A-4. 학생 응용

**A-1 코드 셀에서** `depth_candidates`를 `[1, 2, 3, 4, 5, 7, 10, None]`으로 직접 수정하고 **A-1 → A-2 → A-3** 순서로 다시 실행합니다. 이후 다음 실습 전 기본값으로 되돌릴 수 있습니다.

## 실습 B 준비 — 단일 하이퍼파라미터 그리드 서치

교안 5-2. 실습 B에서 사용할 `gs_single.cv_results_`를 만듭니다. **교안과 동일하게 후보 5개 × 5-폴드 교차 검증**을 수행합니다.

In [ ]:
from sklearn.model_selection import GridSearchCV

params_single = {
    'min_impurity_decrease': [
        0.0001, 0.0002, 0.0003, 0.0004, 0.0005
    ]
}

In [ ]:
gs_single = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    params_single,
    n_jobs=-1
)

gs_single.fit(train_input, train_target)

## 실습 B — GridSearchCV의 후보별 결과 분석하기

교안 5-5. `gs_single`을 다시 학습하지 않고 저장된 교차 검증 결과를 사용합니다.

### B-1. 평균·표준편차·순위 표로 정리

In [ ]:
cv_results = gs_single.cv_results_

results_b = pd.DataFrame({
    'min_impurity_decrease': [
        params['min_impurity_decrease']
        for params in cv_results['params']
    ],
    '평균 검증 정확도': cv_results['mean_test_score'],
    '검증 표준편차': cv_results['std_test_score'],
    '순위': cv_results['rank_test_score']
})

print(results_b.round(4).to_string(index=False))

### B-2. 검증 순위로 정렬

In [ ]:
ranked_b = results_b.sort_values('순위')

print(ranked_b.round(4).to_string(index=False))

### B-3. 1위와 2위의 점수 차이

In [ ]:
mean_scores = cv_results['mean_test_score']

# 평균 검증 정확도를 큰 값부터 정렬
sorted_scores = np.sort(mean_scores)[::-1]

score_difference = sorted_scores[0] - sorted_scores[1]

print(f'최고 평균 검증 정확도: {sorted_scores[0]:.4f}')
print(f'두 번째 평균 검증 정확도: {sorted_scores[1]:.4f}')
print(f'정확도 차이: {score_difference:.4f}')

### B-4. 학생 응용

B-1 결과표에서 **평균 검증 정확도 1위**와 **검증 표준편차 최소** 후보를 각각 확인합니다. 실습 해석과 핵심 질문은 교안 5-5를 참고합니다.

## 실습 마무리

모델 선택에는 **훈련 점수가 아닌 교차 검증 결과**를 사용합니다. 이 노트북에서는 **최종 테스트 정확도를 확인하지 않습니다.**